<a href="https://colab.research.google.com/github/adamabuhamdan/sql-fine-tuning/blob/main/sql-tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Environment Setup

- **transformers**: The core library from Hugging Face for loading models (e.g., Llama) and tokenizers.

- **peft (Parameter-Efficient Fine-Tuning)**: This is the magic here. It contains the LoRA algorithm, which allows us to train the model without needing massive hardware resources.

- **accelerate**: A library that helps manage memory and efficiently distribute workload across the GPU.

- **datasets**: Used to load the dataset that we will train the model on.

- **trl (Transformer Reinforcement Learning)**: Contains the `SFTTrainer` (Supervised Fine-Tuning Trainer), which greatly simplifies the training process.

In [ ]:
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6"

# Section 2: Load Base Model

Here, we fetch the model from the internet and place it onto the GPU to start working with it.

**TinyLlama** was chosen because it is small (around 1.1 billion parameters). This makes it ideal for learning and quick experimentation, as it fits easily within the 16GB memory of a T4 GPU.

**Tokenizer**: This is the translator that converts human text into numbers (tokens) that the model understands.

- **pad_token**: When training the model on texts of different lengths, we need to pad the shorter texts to match the longer ones. If the model does not have its own padding token, we instruct it to use the end-of-sentence token (eos_token) as a substitute.

- **padding_side = "right"**: We place the padding tokens on the right side of the text.

- **dtype = torch.float16**: This line is very important. Models typically load in 32-bit precision. By converting to 16-bit (FP16), we roughly cut memory usage in half with negligible performance loss. Without this line, you may encounter an "Out of Memory" error.

- **device_map = "auto"**: This tells the `accelerate` library to intelligently distribute the model across the available GPU(s).

- **use_cache = False**: Models use caching to speed up text generation, but during fine-tuning, this feature must be disabled because it conflicts with a memory-saving technique called **gradient checkpointing** (which we will see later).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model.config.pretraining_tp = 1




# Section 3: Test the Model Before Fine-Tuning (Inference)

**Goal of this section:**

Before teaching the model a new skill (converting text to SQL code), we must test its current "raw" performance to see how it behaves. This allows us to compare the results later after training.

**`build_prompt`**: We give the model a "persona" (system prompt) as an SQL assistant that must respond with code only. The `apply_chat_template` function is a magic tool in the `transformers` library that formats the conversation using the special tokens that TinyLlama was originally trained on (e.g., `<|system|>`, `<|user|>`).

**`@torch.no_grad()`**: This decorator is very important for an AI engineer. We are telling PyTorch: "I only want inference. Do not calculate and store gradients." This saves a huge amount of memory because it temporarily disables training mode.

**`do_sample=False`**: This means we are using a method called **greedy decoding**, where the model always chooses the word with the highest probability. When writing code, we do not want the model to be "creative" — we want it to be precise.

In [ ]:
# 1. دالة بناء هيكل السؤال (Prompt Template)
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    # هذه الدالة تقوم بتغليف النص بصيغة يفهمها نموذج TinyLlama
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# 2. دالة توليد الإجابات من النموذج
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    # تحويل النص إلى أرقام (Tokens) وإرسالها لكرت الشاشة
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # جعل النموذج يتنبأ بالكلمات القادمة
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False, # نريد إجابة دقيقة وليست إبداعية
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # قص السؤال من النتيجة، والإبقاء على الإجابة الجديدة فقط
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]

    # إعادة تحويل الأرقام إلى نص بشري
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# 3. الأسئلة الاختبارية الثلاثة
PROBES = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("="*70)
print("BASE MODEL (before fine-tuning)")
print("="*70)
base_outputs = []

# تشغيل حلقة (Loop) لسؤال النموذج الأسئلة الثلاثة وطباعة إجاباته
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)

# Section 4: Loading and Preparing the Dataset

The goal here is not merely to fetch data, but to **format** it so that it exactly matches the way the TinyLlama model was originally trained.

## 1. Load and Reduce the Data

The original dataset contains tens of thousands of examples. Using all of them would make training take hours. We shuffle the data to ensure diversity, then take only 3,000 rows as a training sample. After that, we split it into two parts: 95% for training and 5% for evaluation to test the model's performance during training.

## 2. Conversation Structure Engineering (Prompt Engineering)

We reconstruct each row in the dataset to take the form of a conversation (role-play). We define the **system** role, then provide the model with the **table schema**, the **user's question**, and the **ideal assistant response**.

## 3. The Real Magic: `apply_chat_template`

Models do not understand Python dictionaries. They need sequential text containing **special tokens** that tell them when the user's turn starts and ends. The `apply_chat_template` function takes the conversation and wraps it with the tokens that TinyLlama prefers (e.g., `<|system|>`, `<|user|>`). Without this line, the model would learn nonsense.

## 4. Apply the Function and Clean the Data

We pass the `format_example` function to all 3,000 rows. Most importantly, we instruct it to **remove the old columns** because we no longer need them. All that matters now is the `text` column, which contains the fully prepared conversation.

In [ ]:
from datasets import load_dataset

# 1. تحميل مجموعة البيانات من Hugging Face
raw = load_dataset("b-mc2/sql-create-context", split="train")
print("Full dataset size:", len(raw))

# 2. أخذ عينة صغيرة (3000 سطر فقط) لكي لا ننتظر ساعات
raw = raw.shuffle(seed=42).select(range(3000))
split = raw.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Train size:", len(train_ds), " Eval size:", len(eval_ds))

# 3. دالة تغليف البيانات لتصبح بنفس صيغة المحادثة التي يفهمها النموذج
def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    # نضع الـ Schema والسؤال في دور المستخدم
    user      = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    # ونضع الإجابة الصحيحة في دور المساعد الذكي
    assistant = row["answer"]

    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    # تطبيق الـ Template الخاص بـ TinyLlama
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# تطبيق الدالة على كل البيانات
train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map (format_example, remove_columns=eval_ds.column_names)

print("\n--- كيف يبدو شكل المثال بعد التجهيز؟ ---\n")
print(train_ds[0]["text"])

# Section 5: Installing the Magical LoRA Matrices

This section is the heart of **Parameter-Efficient Fine-Tuning (PEFT)**. We will not modify the model's 1.1 billion parameters. Instead, we freeze them and implant very small, trainable "mini-brains" to adjust the model's behavior.

## 1. Smart Memory Management: Gradient Checkpointing

During normal training, mathematical calculations (activations) are stored in memory to be used during backpropagation. This consumes huge amounts of VRAM. By enabling **gradient checkpointing**, we tell the computer: "Do not store these calculations. Recompute them when needed." We sacrifice a little processing time for massive memory savings, preventing Colab from crashing. The second line is technically necessary to allow gradients to pass through to the LoRA matrices while the base model remains frozen.

## 2. Configuring LoRA (`LoraConfig`)

- **`r=16` (Rank)**: This is the size of the new matrix. Think of it as the "bottleneck size." The smaller the value (e.g., 8 or 16), the fewer parameters, but also the lower the capacity. For simple tasks like SQL formatting, `r=16` is excellent.

- **`lora_alpha=32`**: This controls the strength or "voice" of the LoRA matrix over the base model. The general rule of thumb is that alpha is often double the rank (i.e., $16 \times 2 = 32$).

- **`lora_dropout=0.05`**: We randomly turn off 5% of the neurons in LoRA during training. Why? To prevent parrot-like memorization (overfitting) and to force the model to understand the true logic of SQL.

- **`target_modules`**: Where will we implant these matrices in the model's brain? Here, we selected the attention mechanisms: Query (`q_proj`), Key (`k_proj`), Value (`v_proj`), and Output (`o_proj`). These parts are responsible for connecting words and understanding context.

## 3. Wrap and Print the Result

We wrap the LoRA layer around the base model. The final line shows the truth — the number of trainable parameters.

In [ ]:
from peft import LoraConfig, get_peft_model

# تفعيل تقنية لتوفير الذاكرة (Gradient Checkpointing)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # لأن الباراميترز الأساسية مجمدة

# إعدادات LoRA
lora_config = LoraConfig(
    r=16,                                # حجم المصفوفة الجديدة (Rank) - كلما زاد، زاد التعقيد
    lora_alpha=32,                       # قوة تأثير هذه المصفوفات على النموذج
    lora_dropout=0.05,                   # نسبة لتجنب الحفظ الأعمى (Overfitting)
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # الأماكن التي سنزرع فيها المصفوفات
)

# دمج النموذج الأساسي مع مصفوفات LoRA
model = get_peft_model(model, lora_config)

# طباعة عدد الباراميترز القابلة للتدريب
model.print_trainable_parameters()

# Section 6: Starting the Training Process

We are now at the critical stage. We will set up the **Trainer** that will take the data and feed it to the model to adjust the weights of the LoRA matrices.

## The Secret Formula in This Section: Effective Batch Size

The two most important lines in the training settings for those working with limited resources are:

- **`per_device_train_batch_size = 2`**
- **`gradient_accumulation_steps = 8`**

### What does this mean?

The T4 GPU cannot process 16 examples at the same time — the memory would fill up and crash. So we use a clever trick:

1. We feed only **2 examples** at a time (`batch_size = 2`). The model calculates the error (loss), but does **not** update the weights yet.
2. We feed another 2 examples, and so on, repeating this **8 times** (`accumulation_steps = 8`).
3. After the 8 repetitions, the trainer aggregates all the errors and updates the weights **once**.

### The final result:

$2 \times 8 = 16$

The model trains as efficiently as if it had received 16 examples at once, **without burning through the memory!**

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./tinyllama-sql-lora"

# 1. إعدادات التدريب (Hyperparameters)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                  # عدد مرات المرور على البيانات كاملة (دورة واحدة تكفي للتجربة)
    per_device_train_batch_size=2,       # عدد الأمثلة التي تدخل لكرت الشاشة في المرة الواحدة
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,       # تجميع الحسابات لـ 8 خطوات قبل التحديث
    gradient_checkpointing=True,         # تفعيل توفير الذاكرة الذي تحدثنا عنه سابقاً
    learning_rate=2e-4,                  # سرعة التعلم (حجم الخطوة التي يخطوها النموذج للتعديل)
    lr_scheduler_type="cosine",          # طريقة تقليل سرعة التعلم تدريجياً لتثبيت المعلومات
    warmup_steps=10,                     # خطوات إحماء قبل الوصول للسرعة القصوى للتعلم
    optim="adamw_torch",                 # نوع الخوارزمية الرياضية للتحديث
    fp16=True,                           # استخدام دقة 16-bit لتسريع التدريب وتوفير الذاكرة
    bf16=False,                          # كرت T4 لا يدعم bf16 بشكل جيد، لذلك نغلقها
    logging_steps=10,                    # طباعة تقرير عن مستوى الخطأ كل 10 خطوات
    eval_strategy="steps",               # تقييم النموذج بناءً على الخطوات
    eval_steps=50,                       # اختبار النموذج على بيانات التقييم (eval) كل 50 خطوة
    save_strategy="epoch",               # حفظ النموذج في نهاية الدورة
    report_to="none",                    # إغلاق رفع التقارير لمنصات خارجية مثل Weights & Biases
)

# 2. خطوة تقطيع النص (Tokenization) لضمان عدم تجاوز الحد الأقصى
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,        # قص النص إذا كان أطول من اللازم
        max_length=512,         # الحد الأقصى لطول المحادثة المدخلة
        padding=False,
    )
    return out

# تطبيق التقطيع على بيانات التدريب والتقييم
train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map (tokenize, batched=True, remove_columns=["text"])

# 3. إنشاء المدرب (Trainer) وبدء التدريب
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

# هنا يبدأ السحر الفعلي!
trainer.train()

# طباعة أقصى استهلاك للذاكرة بعد التدريب
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

# Section 7: The Moment of Truth (Before vs. After Comparison)

Now it is time to reap the rewards of this training. We will ask the model the same three questions that we asked in Section 3, to see how its behavior has transformed — from a model that rambles and makes mistakes, into a professional, concise SQL programmer.

In [ ]:
# إعادة تفعيل الكاش (Cache) لتسريع توليد النصوص (كنا قد أغلقناه لتوفير الذاكرة أثناء التدريب)
model.config.use_cache = True
# وضع النموذج في وضع التقييم/الاستخدام (Inference Mode)
model.eval()

print("="*70)
print("FINE-TUNED MODEL (after LoRA)")
print("="*70)

# سنقوم بالمرور على نفس الأسئلة (PROBES) مرة أخرى
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)

    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1]) # طباعة الإجابة القديمة (قبل التدريب)
    print("="*70)
    print("AFTER :", ans)               # طباعة الإجابة الجديدة (بعد التدريب)
    print("="*70)

# Section 8 (Final): Save the "Mini-Brain" (Save Adapters)



After confirming that the model is performing excellently, we must save this training so that it is not lost when Google Colab is closed. The advantage here is that we will **not** save the entire model (which would be 2.2 GB). Instead, we will save only the LoRA matrices (the weights that were trained).

In [ ]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"

# حفظ مصفوفات LoRA (الأوزان الجديدة)
model.save_pretrained(ADAPTER_DIR)
# حفظ المترجم (Tokenizer)
tokenizer.save_pretrained(ADAPTER_DIR)

# حساب حجم الملفات المحفوظة على القرص
import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")